In [ ]:
!pip install git+https://github.com/mittagessen/kraken@main

In [ ]:
!kraken get 10.5281/zenodo.2577813

In [ ]:
!find / -name "*.mlmodel" 2>/dev/null

In [ ]:
!pip show kraken

In [ ]:
# Add your own path from you drive
# from google.colab import drive
# drive.mount('/content/drive1')

In [7]:
import os
import torch
import warnings
from tqdm import tqdm
from PIL import Image
from kraken import binarization, pageseg, rpred
from kraken.lib import models, segmentation

In [ ]:
image_folder = "/content/drive1/MyDrive/iamges"
list_file = "/content/list_of_files.txt"    # this is a file containing the names of the images that have ground truth text file(extracted words)
output_folder = "/content/drive1/MyDrive/ocr_results_kraken"
model_path = '/root/.local/share/htrmopo/c895582c-4416-5715-a3e1-4ff1fb2766a6/en_best.mlmodel'  # change this part according to the place that your kraken model is saved with the command written above '!find / -name "*.mlmodel" 2>/dev/null'

In [9]:
warnings.filterwarnings("ignore")

In [ ]:
os.makedirs(output_folder, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.load_any(model_path)
if model.nn is None:
    model.load_model()

with open(list_file, "r", encoding="utf-8") as f:
    raw_names = [line.strip() for line in f if line.strip()]

image_names = []
for name in raw_names:
    if name.lower().endswith(".txt"):
        name = name[:-4]
    if not name.lower().endswith(('.jpg', '.png', '.jpeg', '.tiff')):
        name = name + ".jpg"
    image_names.append(name)

print(f"Found {len(image_names)} images to process.")

for img_name in tqdm(image_names, desc="Processing images"):
    input_path = os.path.join(image_folder, img_name)
    output_path = os.path.join(output_folder, os.path.splitext(img_name)[0] + ".txt")

    if not os.path.exists(input_path):
        print(f"Skipping {img_name}: file not found")
        continue
    if os.path.exists(output_path):
        continue

    img = Image.open(input_path).convert("L")
    img_bin = binarization.nlbin(img)

    seg_result = pageseg.segment(img_bin)

    if not seg_result or (isinstance(seg_result, dict) and "lines" not in seg_result):
        print(f"Skipping {img_name}: segmentation empty or invalid")
        continue

    try:
        predictions = [rec.prediction for rec in rpred.rpred(model, img_bin, seg_result)]
    except Exception as e:
        print(f"Error processing {img_name}: {e}")
        continue

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(predictions))